# Data Preprocessing Contacts – BA Clustering
This notebook reads the raw contact-level receipt data and transforms it into a DataFrame for cluster analysis.

In [579]:
import pandas as pd
import numpy as np
import re
from datetime import datetime, timedelta

## 1. Load Data

In [580]:
# Receipt line items with contact persons
df_receipts_contacts = pd.read_csv('data/Belege_mit_Ansprechpartner.csv', sep=None, engine='python')
print(f'Receipt data: {len(df_receipts_contacts)} rows, {df_receipts_contacts["ADDRESS1PLURAL: Vorname"].nunique()} unique first names')

# Product group definitions
df_product_groups = pd.read_excel('data/Produktgruppen.xlsx')
print(f'Product groups: {df_product_groups.shape}')
df_product_groups.head(3)

Receipt data: 18160 rows, 428 unique first names
Product groups: (176, 8)


,Regale\n10XX - 18XX,Lager Zubehör\n22XX - 27XX + 39XX,Werkstätte\n28XX - 38XX,Werkstätten-Zubehör\n44XX - 48XX,Garderoben\n40XX,Garderoben-Zubehör\n41XX - 42XX,Büroeinrichtungen\n43XX + 50XX - 52XX,Sonstige Produkte
0,1001Z,2286A,2806K,4422A,4000Z,4098A,4322A,1900A
1,1025A,2293A,2825A,4422K,4001K,4099Z,4322K,1900B
2,1030A,2299A,2825B,4422Z,4004A,40AJL,4322Z,191AR


## 2. Build Product Group Mapping

Column headers in Produktgruppen.xlsx contain line breaks (`\n`).
The part before `\n` is the group name used in the final table.

In [581]:
# Mapping: product number prefix -> product group name
product_group_mapping = {}  # {prefix: group_name}
product_group_list = []

for col in df_product_groups.columns:
    # Group name = part before the first \n
    group_name = col.split('\n')[0].strip()
    product_group_list.append(group_name)
    for val in df_product_groups[col].dropna():
        prefix = str(val).strip()
        if prefix:
            product_group_mapping[prefix] = group_name

print(f'Product groups: {product_group_list}')
print(f'Mapped product numbers: {len(product_group_mapping)}')
print('Examples:', dict(list(product_group_mapping.items())[:5]))

Product groups: ['Regale', 'Lager Zubehör', 'Werkstätte', 'Werkstätten-Zubehör', 'Garderoben', 'Garderoben-Zubehör', 'Büroeinrichtungen', 'Sonstige Produkte']
Mapped product numbers: 734
Examples: {'1001Z': 'Regale', '1025A': 'Regale', '1030A': 'Regale', '1030B': 'Regale', '1030C': 'Regale'}


## 3. Preprocess Receipt Data

- Rows without a product number are removed
- The product prefix (part before `_`) is extracted
- Each row is mapped to a product group

In [582]:
# Select relevant columns (contact identifier + receipt data)
df_contact_receipt_items = df_receipts_contacts[['ADDRESS1PLURAL: Vorname', 'ADDRESS1PLURAL: Name', 'ADDRESS1PLURAL: Straße', 'Belege: Datum', 'Belegpositionen: Produktnummer', 'Belegpositionen: Summe']].copy()
df_contact_receipt_items.columns = ['first_name', 'last_name', 'street', 'date', 'product_number', 'revenue']

# Remove rows without a product number
n_before = len(df_contact_receipt_items)
df_contact_receipt_items = df_contact_receipt_items.dropna(subset=['product_number'])
print(f'Rows before filter: {n_before}, after removing NaN product numbers: {len(df_contact_receipt_items)}')

# Convert revenue to numeric (handles European format like "1 201,00" with space as thousands separator)
df_contact_receipt_items['revenue'] = (
    df_contact_receipt_items['revenue']
    .astype(str)
    .str.replace('\xa0', '', regex=False)   # non-breaking space
    .str.replace(' ', '', regex=False)       # regular space (thousands separator)
    .str.replace(',', '.', regex=False)      # decimal comma → point
    .pipe(pd.to_numeric, errors='coerce')
)

# Parse dates
df_contact_receipt_items['date'] = pd.to_datetime(df_contact_receipt_items['date'], dayfirst=True)

# Extract product prefix: part before "_"
df_contact_receipt_items['product_prefix'] = df_contact_receipt_items['product_number'].astype(str).str.split('_').str[0].str.strip()

# Map each product prefix to its product group
df_contact_receipt_items['product_group'] = df_contact_receipt_items['product_prefix'].map(product_group_mapping).fillna('Sonstige Produkte')

n_unmapped = df_contact_receipt_items['product_group'].isna().sum()
print(f'Unmapped product numbers: {n_unmapped} ({n_unmapped / len(df_contact_receipt_items) * 100:.1f}%)')
if n_unmapped > 0:
    print('Examples:', df_contact_receipt_items[df_contact_receipt_items['product_group'].isna()]['product_prefix'].value_counts().head(10).to_dict())

df_contact_receipt_items.head()

Rows before filter: 18160, after removing NaN product numbers: 14384
Unmapped product numbers: 649 (4.5%)
Examples: {'0': 392, '9953FRACHT': 172, '9957FRACHT': 31, '9968FRACHT': 19, '9962FRACHT': 10, 'Stk': 10, '9933FRACHT': 5, '9999FRACHT': 4, '9946FRACHT': 1, '5': 1}


,first_name,last_name,street,date,product_number,revenue,product_prefix,product_group
0,Daniel W.,Beckler,Zagrebačka cesta 22,2025-10-24 13:15:38,48ACA_12350,1680.0,48ACA,Werkstätten-Zubehör
1,Daniel W.,Beckler,Zagrebačka cesta 22,2025-10-24 13:15:38,99ACFRACHT_12346,0.0,99ACFRACHT,Sonstige Produkte
3,Daniel W.,Beckler,Zagrebačka cesta 22,2025-10-24 13:15:38,1130C1_17242,1201.0,1130C1,Regale
4,Daniel W.,Beckler,Zagrebačka cesta 22,2025-10-24 13:15:38,1130C1_17243,1743.0,1130C1,Regale
5,Daniel W.,Beckler,Zagrebačka cesta 22,2025-10-24 13:15:38,1130C1_17246,837.0,1130C1,Regale


In [583]:
df_contact_receipt_items["product_group"].value_counts()

product_group
Werkstätte             4003
Regale                 3501
Sonstige Produkte      1923
Garderoben             1761
Garderoben-Zubehör     1352
Werkstätten-Zubehör     785
Lager Zubehör           324
Büroeinrichtungen        86
Name: count, dtype: int64

## 4. Calculate RFM Features

- **Recency**: days since the most recent receipt per contact
- **Frequency**: number of unique purchase dates in the last 3 years per contact
- **Product Group Revenue**: total revenue per product group per contact (all time)

In [584]:
# Reference date for the "last 3 years" window
reference_date = pd.Timestamp.today().normalize()
cutoff_date = reference_date - pd.DateOffset(years=3)
print(f'Reference date: {reference_date.date()}, 3-year cutoff: {cutoff_date.date()}')

# --- Recency ---
# Most recent purchase date per contact (using all receipt data)
df_contact_purchase_dates = df_receipts_contacts[['ADDRESS1PLURAL: Vorname', 'ADDRESS1PLURAL: Name', 'ADDRESS1PLURAL: Straße', 'Belege: Datum']].copy()
df_contact_purchase_dates.columns = ['first_name', 'last_name', 'street', 'date']
df_contact_purchase_dates['date'] = pd.to_datetime(df_contact_purchase_dates['date'], dayfirst=True)
recency_contacts = df_contact_purchase_dates.groupby(['first_name', 'last_name', 'street'])['date'].max().rename('Recency')

# --- Frequency ---
# Number of unique purchase dates per contact within the last 3 years
df_contact_purchases_last_3_years = df_contact_purchase_dates[df_contact_purchase_dates['date'] >= cutoff_date]
frequency_contacts = (
    df_contact_purchases_last_3_years.groupby(['first_name', 'last_name', 'street'])['date']
    .nunique()
    .rename('Frequency')
)

print(f'Recency: {len(recency_contacts)} contacts')
print(f'Frequency: {len(frequency_contacts)} contacts with purchases in last 3 years')

Reference date: 2026-04-04, 3-year cutoff: 2023-04-04
Recency: 1214 contacts
Frequency: 674 contacts with purchases in last 3 years


In [585]:
# --- Product Group Revenue ---
# Only use rows that were successfully mapped to a product group
df_contact_receipt_items_mapped = df_contact_receipt_items.dropna(subset=['product_group'])

# Sum revenue per contact and product group
pg_revenue_contacts = (
    df_contact_receipt_items_mapped
    .groupby(['first_name', 'last_name', 'street', 'product_group'])['revenue']
    .sum()
    .unstack(fill_value=0)
)

# Ensure all product groups exist as columns (even if no contact bought from them)
for group in product_group_list:
    if group not in pg_revenue_contacts.columns:
        pg_revenue_contacts[group] = 0

# Reorder columns to match the defined group order
pg_revenue_contacts = pg_revenue_contacts[product_group_list]

print('Product group revenue shape:', pg_revenue_contacts.shape)
pg_revenue_contacts.head()

Product group revenue shape: (1193, 8)


,,product_group,Regale,Lager Zubehör,Werkstätte,Werkstätten-Zubehör,Garderoben,Garderoben-Zubehör,Büroeinrichtungen,Sonstige Produkte
first_name,last_name,street,,,,,,,,
Adem,Yilmaz,"Erdbergstraße 202, PF 63",135.0,0.0,0.0,0.0,5273.0,0.0,0.0,1020.0
Adiel,Fuks,Simon-Wiesenthal-Gasse 5,0.0,0.0,0.0,3128.8,0.0,0.0,0.0,0.0
Adile,Gülhan,Lassallestraße 5,0.0,0.0,107.8,0.0,318.0,0.0,0.0,12.0
Adis,Selimovic,Rasumofskygasse 2,0.0,0.0,0.0,0.0,456.0,0.0,0.0,0.0
Admir,Zehic,Ignaz-Köck-Straße 8,0.0,0.0,0.0,451.0,0.0,0.0,0.0,0.0


## 5. Group receipts by contacts

In [586]:
# Start with all unique contacts from the receipts dataset (identified by first_name + last_name + street)
all_contacts = df_receipts_contacts[['ADDRESS1PLURAL: Vorname', 'ADDRESS1PLURAL: Name', 'ADDRESS1PLURAL: Straße']].drop_duplicates()
all_contacts.columns = ['first_name', 'last_name', 'street']
df_contacts = all_contacts.set_index(['first_name', 'last_name', 'street'])

# Join Recency and Frequency
df_contacts = df_contacts.join(recency_contacts, how='left')
df_contacts = df_contacts.join(frequency_contacts, how='left')
df_contacts['Frequency'] = df_contacts['Frequency'].fillna(0).astype(int)

# Join product group revenue
df_contacts = df_contacts.join(pg_revenue_contacts, how='left')
df_contacts[product_group_list] = df_contacts[product_group_list].fillna(0)

# Calculate total revenue as sum of all product group revenues
df_contacts['Monetary Value'] = df_contacts[product_group_list].sum(axis=1)

# Reset index
df_contacts = df_contacts.reset_index()

print(f'Final table: {df_contacts.shape}')
df_contacts.head(10)

Final table: (1265, 14)


,first_name,last_name,street,Recency,Frequency,Regale,Lager Zubehör,Werkstätte,Werkstätten-Zubehör,Garderoben,Garderoben-Zubehör,Büroeinrichtungen,Sonstige Produkte,Monetary Value
0,Daniel W.,Beckler,Zagrebačka cesta 22,2025-10-24 13:15:38,1,8367.0,0.0,0.0,1680.0,0.0,0.0,0.0,0.0,10047.0
1,Davor,Krcmar,Zagrebačka cesta 22,2022-04-26 12:04:56,0,804.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,804.0
2,Domen,Smigoc,Zagrebačka cesta 22,2021-11-08 11:32:17,0,510.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,510.0
3,Marko,Koprivnik,Zagrebačka cesta 22,2021-07-22 11:30:46,0,1202.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1202.0
4,Matjaz,Friedl,Zagrebačka cesta 22,2025-11-18 15:33:25,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,Petra,Muren,Zagrebačka cesta 22,2025-11-18 15:38:07,1,8367.0,0.0,0.0,1680.0,0.0,0.0,0.0,0.0,10047.0
6,Cornelia,Gänger,Leobersdorfer Straße 26,2023-09-08 11:33:32,1,0.0,0.0,527.0,0.0,0.0,0.0,0.0,0.0,527.0
7,Franz,Kager,Leobersdorfer Straße 26,2020-09-11 12:29:01,0,0.0,0.0,39.0,0.0,0.0,0.0,0.0,0.0,39.0
8,Harald,Steurer,Leobersdorfer Straße 26,2021-03-16 15:57:30,0,85.4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,85.4
9,Jean,Dusabe,Leobersdorfer Straße 26,2024-11-20 12:55:17,1,0.0,0.0,0.0,0.0,0.0,139.9,0.0,0.0,139.9


In [587]:
# Overview
print('Columns:', df_contacts.columns.tolist())
print()
print(df_contacts.describe())

Columns: ['first_name', 'last_name', 'street', 'Recency', 'Frequency', 'Regale', 'Lager Zubehör', 'Werkstätte', 'Werkstätten-Zubehör', 'Garderoben', 'Garderoben-Zubehör', 'Büroeinrichtungen', 'Sonstige Produkte', 'Monetary Value']

                             Recency    Frequency         Regale  \
count                           1214  1265.000000    1265.000000   
mean   2023-06-04 03:44:09.208401920     1.111462    2621.672909   
min              2019-09-10 16:08:25     0.000000    -883.260000   
25%              2021-10-04 22:09:18     0.000000       0.000000   
50%       2023-09-06 12:12:32.500000     1.000000       0.000000   
75%    2025-02-28 00:15:14.249999872     1.000000    1602.000000   
max              2026-03-13 13:23:36    38.000000  102597.000000   
std                              NaN     3.136313    8265.115767   

       Lager Zubehör     Werkstätte  Werkstätten-Zubehör     Garderoben  \
count    1265.000000    1265.000000          1265.000000    1265.000000   
mean 

## 6. Contact Enrichment – Join with Ansprechpartner.csv

Left join on `first_name`, `last_name`, `street` to enrich contacts with department, industry, company size, type detail, and Bundesland.

In [588]:
# Load Alle_Ansprechpartner.csv (company sizes already pre-filled in fake_company_size.ipynb)
df_ansprechpartner_original = pd.read_csv('data/Alle_Ansprechpartner.csv', sep=None, engine='python')
print(f'Ansprechpartner data: {len(df_ansprechpartner_original)} rows')

# Select relevant columns for join
df_ansprechpartner = df_ansprechpartner_original[['CHRISTIANNAME', 'NAME', 'STREET1', 'DEPARTMENT', 'GWBRANCH', 'OS_MA_ANZ_GRUP', 'COMPNAME', 'GWSTATE1']].copy()
df_ansprechpartner.columns = ['first_name', 'last_name', 'street', 'department', 'industry', 'company_size', 'company_name', 'bundesland']

# Drop duplicates on the join key (first_name + last_name + street)
df_ansprechpartner = df_ansprechpartner.drop_duplicates(subset=['first_name', 'last_name', 'street'])
print(f'Unique contacts in Ansprechpartner: {len(df_ansprechpartner)}')

# Left join: keep all contacts from df_contacts, enrich with Ansprechpartner data
df_contacts = df_contacts.merge(
    df_ansprechpartner,
    on=['first_name', 'last_name', 'street'],
    how='left'
)

n_matched = df_contacts['bundesland'].notna().sum()
print(f'Contacts matched with Ansprechpartner: {n_matched} / {len(df_contacts)} ({n_matched / len(df_contacts) * 100:.1f}%)')
df_contacts.head()

Ansprechpartner data: 6826 rows
Unique contacts in Ansprechpartner: 6665
Contacts matched with Ansprechpartner: 1194 / 1265 (94.4%)


,first_name,last_name,street,Recency,Frequency,Regale,Lager Zubehör,Werkstätte,Werkstätten-Zubehör,Garderoben,Garderoben-Zubehör,Büroeinrichtungen,Sonstige Produkte,Monetary Value,department,industry,company_size,type_detail,company_name,bundesland
0,Daniel W.,Beckler,Zagrebačka cesta 22,2025-10-24 13:15:38,1,8367.0,0.0,0.0,1680.0,0.0,0.0,0.0,0.0,10047.0,NaN,NaN,NaN,NaN,NaN,NaN
1,Davor,Krcmar,Zagrebačka cesta 22,2022-04-26 12:04:56,0,804.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,804.0,NaN,NaN,NaN,NaN,NaN,NaN
2,Domen,Smigoc,Zagrebačka cesta 22,2021-11-08 11:32:17,0,510.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,510.0,NaN,NaN,NaN,NaN,NaN,NaN
3,Marko,Koprivnik,Zagrebačka cesta 22,2021-07-22 11:30:46,0,1202.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1202.0,NaN,NaN,NaN,NaN,NaN,NaN
4,Matjaz,Friedl,Zagrebačka cesta 22,2025-11-18 15:33:25,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN


In [589]:
print(df_contacts.isna().sum())

first_name              44
last_name                4
street                   3
Recency                 51
Frequency                0
Regale                   0
Lager Zubehör            0
Werkstätte               0
Werkstätten-Zubehör      0
Garderoben               0
Garderoben-Zubehör       0
Büroeinrichtungen        0
Sonstige Produkte        0
Monetary Value           0
department             151
industry                64
company_size            64
type_detail            116
company_name            64
bundesland              71
dtype: int64


In [590]:
# Remove rows where (first_name AND Recency are both NaN) OR (bundesland is NaN)
name_and_recency_missing = df_contacts["first_name"].isna() & df_contacts["Recency"].isna()
bundesland_missing = (
    df_contacts["bundesland"].isna()
)

rows_to_remove = name_and_recency_missing | bundesland_missing
n_removed = rows_to_remove.sum()
print(f"Rows removed: {n_removed}")

df_contacts = df_contacts[~rows_to_remove].reset_index(drop=True)
print(f"Remaining contacts: {len(df_contacts)}")

Rows removed: 107
Remaining contacts: 1158


In [591]:
df_contacts.to_csv("data/df_contacts_after_join_in_6.csv")
print(df_contacts.isna().sum())

first_name              0
last_name               2
street                  1
Recency                 3
Frequency               0
Regale                  0
Lager Zubehör           0
Werkstätte              0
Werkstätten-Zubehör     0
Garderoben              0
Garderoben-Zubehör      0
Büroeinrichtungen       0
Sonstige Produkte       0
Monetary Value          0
department             75
industry                0
company_size            0
type_detail            44
company_name            0
bundesland              0
dtype: int64


In [592]:
print(df_contacts.isna().sum())
df_contacts[df_contacts["Recency"].isna()]

first_name              0
last_name               2
street                  1
Recency                 3
Frequency               0
Regale                  0
Lager Zubehör           0
Werkstätte              0
Werkstätten-Zubehör     0
Garderoben              0
Garderoben-Zubehör      0
Büroeinrichtungen       0
Sonstige Produkte       0
Monetary Value          0
department             75
industry                0
company_size            0
type_detail            44
company_name            0
bundesland              0
dtype: int64


,first_name,last_name,street,Recency,Frequency,Regale,Lager Zubehör,Werkstätte,Werkstätten-Zubehör,Garderoben,Garderoben-Zubehör,Büroeinrichtungen,Sonstige Produkte,Monetary Value,department,industry,company_size,type_detail,company_name,bundesland
39,Michael Peter,NaN,Simone-de-Beauvoir-Platz 6,NaT,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Technik / Verkauf / Vertrieb,ZG 7 öffentl. Verwaltung - Versorger - Entsorg...,51-350 Mittelunternehmen,Gewerbl. Nutzer,"Magistrat der Stadt Wien, MA 54",Wien
457,Aleksandar,Orlic,NaN,NaT,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,Telenova Lead,51-350 Mittelunternehmen,NaN,Wolfsforschungszentrum GmbH,Niederösterreich
605,Darek,NaN,Eckertstraße 30M/1/08,NaT,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Service / Kundendienst / Montage,ZG 4 Handel/Gewerbe mit Service - Montage auße...,1-10 Kleinstunternehmen,Wiederverkäufer,TB Shopteam GmbH,Steiermark


In [593]:
df_contacts = df_contacts.dropna(subset=["Recency"])

In [594]:
print(df_contacts.isna().sum())


first_name              0
last_name               0
street                  0
Recency                 0
Frequency               0
Regale                  0
Lager Zubehör           0
Werkstätte              0
Werkstätten-Zubehör     0
Garderoben              0
Garderoben-Zubehör      0
Büroeinrichtungen       0
Sonstige Produkte       0
Monetary Value          0
department             74
industry                0
company_size            0
type_detail            43
company_name            0
bundesland              0
dtype: int64


## 7. Feature Engineering for Clustering

- `industry_group` is simplified into merged groups (ZG 1+2, ZG 3+4, ZG 5+6, ZG 7, ZG 8, ZG 9); rows with `Sonstige` are removed
- Product groups are merged: Werkstätte + Werkstätten-Zubehör → `Werkstätte_gesamt`, Garderoben + Garderoben-Zubehör → `Garderoben_gesamt`, Büroeinrichtungen → `Sonstige Produkte`
- Company sizes are simplified: Klein, Mittel, Groß
- Bundesländer are grouped into regions: Ost, Süd, West
- `department` NaN → `Allgemeine Verwaltung`, then one-hot encoded
- `Recency` is converted to days since the most recent purchase
- Contact names are replaced by a numeric `contact_id`
- Categorical columns are one-hot encoded
- All numeric features are standardised with z-score (StandardScaler)

In [595]:
# Remove contacts without Bundesland
df_contacts_without_location = df_contacts[df_contacts['bundesland'].isna()]
print(f'Contacts without Bundesland (removed): {len(df_contacts_without_location)}')

df_contacts = df_contacts.dropna(subset=['bundesland']).reset_index(drop=True)
print(f'Remaining contacts: {len(df_contacts)}')

Contacts without Bundesland (removed): 0
Remaining contacts: 1155


In [596]:
print(df_contacts['company_size'].value_counts(dropna=False).to_string())

company_size
51-350 Mittelunternehmen          515
351-1000 Mittelgroßunternehmen    300
11-50 Kleinunternehmen            169
>1000 Großunternehmen             143
1-10 Kleinstunternehmen            28


In [597]:
print(df_contacts['industry'].value_counts().to_string())

industry
ZG 7 öffentl. Verwaltung - Versorger - Entsorger - Gesundheit - Schulen - Kultur | Gesundheitswesen | Krankenhäuser                                                                                                           104
ZG 7 öffentl. Verwaltung - Versorger - Entsorger - Gesundheit - Schulen - Kultur | Bildung Personen | Universitäten und Hochschulen                                                                                            57
ZG 7 öffentl. Verwaltung - Versorger - Entsorger - Gesundheit - Schulen - Kultur                                                                                                                                               50
ZG 7 öffentl. Verwaltung - Versorger - Entsorger - Gesundheit - Schulen - Kultur | öffentl. Verwaltung | Stadt- und Gemeindeverwaltung                                                                                         37
ZG 7 öffentl. Verwaltung - Versorger - Entsorger - Gesundheit - Schulen - Kultur | Sich

In [598]:
def map_to_industry_group(industry):
    """Maps raw industry strings directly to the final reduced industry groups."""
    if pd.isna(industry):
        return 'Sonstige'
    
    # Extract all ZG numbers mentioned
    zg_numbers = re.findall(r'ZG\s*(\d+)', str(industry))
    
    # Filter out 99
    valid_zg = [int(n) for n in zg_numbers if n != '99']
    
    if not valid_zg:
        return 'Sonstige'
    
    first_zg = valid_zg[0]
    
    # Map directly to final groups
    if first_zg in (1, 2):
        return 'ZG 1+2'
    elif first_zg in (3, 4):
        return 'ZG 3+4'
    elif first_zg in (5, 6):
        return 'ZG 5+6'
    elif first_zg == 7:
        return 'ZG 7'
    elif first_zg == 8:
        return 'ZG 8'
    elif first_zg == 9:
        return 'ZG 9'
    else:
        return 'Sonstige'

df_contacts['industry_group'] = df_contacts['industry'].apply(map_to_industry_group)

print('Industry groups (final):')
print(df_contacts['industry_group'].value_counts().to_string())

Industry groups (final):
industry_group
ZG 7        567
ZG 1+2      238
ZG 5+6      105
ZG 8         94
ZG 3+4       63
ZG 9         59
Sonstige     29


In [599]:
print(df_contacts["bundesland"].value_counts())

bundesland
Wien                 689
Niederösterreich     292
Oberösterreich        57
Steiermark            52
Burgenland            38
Baden-Württemberg     11
Salzburg               5
Tirol                  3
Kärnten                3
Vorarlberg             2
Bayern                 2
Podravska              1
Name: count, dtype: int64


In [600]:
# Map bundesland to regions (Ost / Süd / West)

bundesland_to_region_mapping = {
    'Wien': 'Ost',
    'Niederösterreich': 'Ost',
    'Burgenland': 'Ost',
    'Salzburg': 'Süd',
    'Kärnten': 'Süd',
    'Steiermark': 'Süd',
    'Oberösterreich': 'West',
    'Tirol': 'West',
    'Vorarlberg': 'West',
}

# Everything not in the mapping (foreign countries etc.) becomes 'West'
df_contacts['bundesland'] = df_contacts['bundesland'].map(bundesland_to_region_mapping).fillna('West')

print('Bundesland regions (final):')
print(df_contacts['bundesland'].value_counts().to_string())

Bundesland regions (final):
bundesland
Ost     1019
West      76
Süd       60


In [601]:
# Department: fill NaN with default, then group into 4 categories
df_contacts["department"] = df_contacts["department"].fillna("Allgemeine Verwaltung")

department_group_mapping = {
    "Einkauf": "Einkauf & Beschaffung",
    "Materialwirtschaft": "Einkauf & Beschaffung",
    "Lager / Logistik intern": "Einkauf & Beschaffung",
    "Facility Management / Instandhaltung": "Technik & Produktion",
    "Produktion / Werkstätte": "Technik & Produktion",
    "Forschung / Entwicklung / Arbeitsvorbereitung": "Technik & Produktion",
    "Forschung / Entwicklung": "Technik & Produktion",
    "Montageleitung": "Technik & Produktion",
    "Haustechnik Elektro": "Technik & Produktion",
    "Technische Leitung / Innendienst": "Technik & Produktion",
    "Technischer Innendienst": "Technik & Produktion",
    "Abteilung Elektro-Und Maschinentechnik": "Technik & Produktion",
    "Service / Kundendienst / Montage": "Technik & Produktion",
    "Sicherheitsbeauftragter": "Technik & Produktion",
    "Lehrlingsausbildung": "Technik & Produktion",
    "Allgemeine Verwaltung": "Verwaltung & Vertrieb",
    "Organisation / Office": "Verwaltung & Vertrieb",
    "Organisation / Verkauf": "Verwaltung & Vertrieb",
    "Technik / Verkauf / Vertrieb": "Verwaltung & Vertrieb",
    "Andere": "Verwaltung & Vertrieb",
    "Geschäftsführung": "Führung & Leitung",
    "Geschäftsführer / Außendienst": "Führung & Leitung",
    "Betriebsleitung": "Führung & Leitung",
    "Projektentwicklung": "Führung & Leitung",
    "Bürgermeister": "Führung & Leitung",
}

df_contacts["department"] = df_contacts["department"].map(department_group_mapping)

print("Department groups (final):")
print(df_contacts["department"].value_counts().to_string())

Department groups (final):
department
Einkauf & Beschaffung    314
Technik & Produktion     302
Führung & Leitung        292
Verwaltung & Vertrieb    247


In [602]:
# (type_detail removed from analysis)

Type detail categories:
type_detail
Gewerbl. Nutzer    1066
Wiederverkäufer      58
Multiplikator        31


In [603]:
# --- Save raw df_contacts before any transformation ---
df_contacts.to_csv('data/df_contacts_final_before_encoding_and_scaling.csv', index=False)
print(f'Saved raw df_contacts: {df_contacts.shape}')
print(f'Columns: {df_contacts.columns.tolist()}')

Saved raw df_contacts: (1155, 21)
Columns: ['first_name', 'last_name', 'street', 'Recency', 'Frequency', 'Regale', 'Lager Zubehör', 'Werkstätte', 'Werkstätten-Zubehör', 'Garderoben', 'Garderoben-Zubehör', 'Büroeinrichtungen', 'Sonstige Produkte', 'Monetary Value', 'department', 'industry', 'company_size', 'type_detail', 'company_name', 'bundesland', 'industry_group']


In [604]:
# Company sizes were already pre-filled in fake_company_size.ipynb
print(f'Contacts with missing company size: {df_contacts["company_size"].isna().sum()}')
print(f'\ncompany_size distribution:')
print(df_contacts['company_size'].value_counts(dropna=False).to_string())

Contacts with missing company size: 0

company_size distribution:
company_size
51-350 Mittelunternehmen          515
351-1000 Mittelgroßunternehmen    300
11-50 Kleinunternehmen            169
>1000 Großunternehmen             143
1-10 Kleinstunternehmen            28


In [605]:
# --- Recency: Datum → Anzahl Tage seit letztem Kauf ---
reference_date = pd.Timestamp.today().normalize()
df_contacts['Recency'] = (reference_date - pd.to_datetime(df_contacts['Recency'])).dt.days
print(f'Recency (days) – min: {df_contacts["Recency"].min()}, max: {df_contacts["Recency"].max()}, mean: {df_contacts["Recency"].mean():.1f}')

# --- Mapping speichern: Index (contact_id) → Ansprechpartner ---
df_contact_id_mapping = df_contacts[['first_name', 'last_name', 'street', 'company_name']].copy()
df_contact_id_mapping.index.name = 'contact_id'
df_contact_id_mapping.to_excel('data/contact_id_mapping.xlsx')

# Remove identity columns – the index serves as contact_id
df_contacts = df_contacts.drop(columns=['first_name', 'last_name', 'street', 'company_name'])

print(f'\nContact ID mapping saved ({len(df_contact_id_mapping)} rows).')
df_contacts.head(3)

Recency (days) – min: 21, max: 2397, mean: 1039.5

Contact ID mapping saved (1155 rows).


,Recency,Frequency,Regale,Lager Zubehör,Werkstätte,Werkstätten-Zubehör,Garderoben,Garderoben-Zubehör,Büroeinrichtungen,Sonstige Produkte,Monetary Value,department,industry,company_size,type_detail,bundesland,industry_group
0,938,1,0.0,0.0,527.0,0.0,0.0,0.0,0.0,0.0,527.0,Einkauf & Beschaffung,ZG 2 Industrie/Produktion mit Betriebseinrichu...,51-350 Mittelunternehmen,Gewerbl. Nutzer,Ost,ZG 1+2
1,2030,0,0.0,0.0,39.0,0.0,0.0,0.0,0.0,0.0,39.0,Führung & Leitung,ZG 2 Industrie/Produktion mit Betriebseinrichu...,51-350 Mittelunternehmen,Gewerbl. Nutzer,Ost,ZG 1+2
2,1844,0,85.4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,85.4,Technik & Produktion,ZG 2 Industrie/Produktion mit Betriebseinrichu...,51-350 Mittelunternehmen,Gewerbl. Nutzer,Ost,ZG 1+2


In [606]:
# --- Dimension Reduction & Encoding ---

# Drop raw industry column (keep industry_group which is already in final groups)
df_contacts = df_contacts.drop(columns=['industry'])

# === 1. Merge product groups ===
df_contacts['Werkstätte_gesamt'] = df_contacts['Werkstätte'] + df_contacts['Werkstätten-Zubehör']
df_contacts['Garderoben_gesamt'] = df_contacts['Garderoben'] + df_contacts['Garderoben-Zubehör']
df_contacts['Sonstige Produkte'] = df_contacts['Sonstige Produkte'] + df_contacts['Büroeinrichtungen']

df_contacts = df_contacts.drop(columns=[
    'Werkstätte', 'Werkstätten-Zubehör',
    'Garderoben', 'Garderoben-Zubehör',
    'Büroeinrichtungen'
])

product_group_columns = ['Regale', 'Lager Zubehör', 'Werkstätte_gesamt', 'Garderoben_gesamt', 'Sonstige Produkte']
print('=== Product groups after merging ===')
print(f'Columns: {product_group_columns}')
print(df_contacts[product_group_columns].describe().round(1))

# === 2. Merge company size ===
company_size_mapping = {
    '1-10 Kleinstunternehmen': 'Klein',
    '11-50 Kleinunternehmen': 'Klein',
    '51-350 Mittelunternehmen': 'Mittel',
    '351-1000 Mittelgroßunternehmen': 'Groß',
    '>1000 Großunternehmen': 'Groß',
}
df_contacts['company_size'] = df_contacts['company_size'].map(company_size_mapping)
print('\n=== Company size after merging ===')
print(df_contacts['company_size'].value_counts(dropna=False).to_string())

# === 3. Remove rows with "Sonstige" industry ===
n_before_sonstige_removal = len(df_contacts)
df_contacts = df_contacts.dropna(subset=['industry_group'])
df_contacts = df_contacts[df_contacts['industry_group'] != 'Sonstige']
n_removed_sonstige = n_before_sonstige_removal - len(df_contacts)
print(f'\n=== Removed {n_removed_sonstige} rows with "Sonstige" industry ===')
print(f'Remaining contacts: {len(df_contacts)}')
print(df_contacts['industry_group'].value_counts().to_string())

# === 4. Update contact ID mapping (without Sonstige) ===
df_original_mapping = pd.read_excel('data/contact_id_mapping.xlsx', index_col='contact_id')
df_contact_id_mapping_final = df_original_mapping.loc[df_contacts.index]
df_contact_id_mapping_final.to_excel('data/contact_id_mapping.xlsx')
print(f'\n=== Updated contact ID mapping: {len(df_contact_id_mapping_final)} contacts ===')

# === 5. One-Hot Encoding ===
categorical_columns = ['industry_group', 'bundesland', 'company_size', 'department']
df_contacts = pd.get_dummies(df_contacts, columns=categorical_columns, dtype=int)

print(f'\n=== Final shape after one-hot encoding: {df_contacts.shape} ===')
print(f'Columns: {df_contacts.columns.tolist()}')

# === 6. Save unscaled data (for cluster interpretation later) ===
df_contacts.to_csv('data/df_contacts_final_before_scaler.csv', index_label='contact_id')
print(f'\nSaved df_contacts_final_before_scaler.csv: {df_contacts.shape}')

=== Product groups after merging ===
Columns: ['Regale', 'Lager Zubehör', 'Werkstätte_gesamt', 'Garderoben_gesamt', 'Sonstige Produkte']
         Regale  Lager Zubehör  Werkstätte_gesamt  Garderoben_gesamt  \
count    1155.0         1155.0             1155.0             1155.0   
mean     2745.1          133.7             3541.8             4658.5   
std      8551.3         1003.2            17443.2            12932.6   
min      -883.3            0.0                0.0           -27249.4   
25%         0.0            0.0                0.0                0.0   
50%         0.0            0.0                0.0                0.0   
75%      1695.6            0.0              570.0             2387.0   
max    102597.0        23009.6           209903.9           139138.0   

       Sonstige Produkte  
count             1155.0  
mean              1159.4  
std               3623.4  
min             -27840.0  
25%                  0.0  
50%                  0.0  
75%               1080.0 

In [607]:
from sklearn.preprocessing import StandardScaler

# Alle numerischen Spalten skalieren
numeric_cols = df_contacts.select_dtypes(include='number').columns.tolist()

scaler = StandardScaler()
df_contacts_final = df_contacts.copy()
df_contacts_final[numeric_cols] = scaler.fit_transform(df_contacts[numeric_cols])

print(f'Scaled {len(numeric_cols)} numeric columns.')
print(df_contacts_final[numeric_cols].describe().round(3))
df_contacts_final.head(3)

Scaled 27 numeric columns.
        Recency  Frequency    Regale  Lager Zubehör  Sonstige Produkte  \
count  1126.000   1126.000  1126.000       1126.000           1126.000   
mean     -0.000      0.000     0.000          0.000             -0.000   
std       1.000      1.000     1.000          1.000              1.000   
min      -1.437     -0.360    -0.422         -0.133             -7.957   
25%      -0.908     -0.360    -0.317         -0.133             -0.319   
50%      -0.124     -0.040    -0.317         -0.133             -0.316   
75%       0.864     -0.040    -0.124         -0.133             -0.023   
max       1.934     11.821    11.849         22.573             11.022   

       Monetary Value  Werkstätte_gesamt  Garderoben_gesamt  \
count        1126.000           1126.000           1126.000   
mean            0.000              0.000             -0.000   
std             1.000              1.000              1.000   
min            -1.707             -0.200             -

,Recency,Frequency,Regale,Lager Zubehör,Sonstige Produkte,Monetary Value,Werkstätte_gesamt,Garderoben_gesamt,industry_group_ZG 1+2,industry_group_ZG 3+4,...,company_size_Groß,company_size_Klein,company_size_Mittel,department_Einkauf & Beschaffung,department_Führung & Leitung,department_Technik & Produktion,department_Verwaltung & Vertrieb,type_detail_Gewerbl. Nutzer,type_detail_Multiplikator,type_detail_Wiederverkäufer
0,-0.135304,-0.039859,-0.317468,-0.132678,-0.319277,-0.456871,-0.168865,-0.359903,1.931604,-0.243447,...,-0.803865,-0.444828,1.122962,1.611659,-0.587606,-0.602658,-0.498334,0.292958,-0.168257,-0.233039
1,1.414632,-0.360435,-0.317468,-0.132678,-0.319277,-0.476266,-0.198101,-0.359903,1.931604,-0.243447,...,-0.803865,-0.444828,1.122962,-0.620479,1.701821,-0.602658,-0.498334,0.292958,-0.168257,-0.233039
2,1.150632,-0.360435,-0.307340,-0.132678,-0.319277,-0.474422,-0.200437,-0.359903,1.931604,-0.243447,...,-0.803865,-0.444828,1.122962,-0.620479,-0.587606,1.659317,-0.498334,0.292958,-0.168257,-0.233039


In [608]:
# --- Finalen Datensatz speichern ---
df_contacts_final.to_csv('data/df_contacts_final.csv', index_label='contact_id')
print(f'Saved df_contacts_final: {df_contacts_final.shape}')
print('Columns:', df_contacts_final.columns.tolist())

Saved df_contacts_final: (1126, 27)
Columns: ['Recency', 'Frequency', 'Regale', 'Lager Zubehör', 'Sonstige Produkte', 'Monetary Value', 'Werkstätte_gesamt', 'Garderoben_gesamt', 'industry_group_ZG 1+2', 'industry_group_ZG 3+4', 'industry_group_ZG 5+6', 'industry_group_ZG 7', 'industry_group_ZG 8', 'industry_group_ZG 9', 'bundesland_Ost', 'bundesland_Süd', 'bundesland_West', 'company_size_Groß', 'company_size_Klein', 'company_size_Mittel', 'department_Einkauf & Beschaffung', 'department_Führung & Leitung', 'department_Technik & Produktion', 'department_Verwaltung & Vertrieb', 'type_detail_Gewerbl. Nutzer', 'type_detail_Multiplikator', 'type_detail_Wiederverkäufer']
